<a href="https://colab.research.google.com/github/KoAlbert/YOLOv1-ResNet-Multitasking-Joint-Detection-and-Segmentation-free-space-lane-marking--BDD100K/blob/main/%E3%80%8CYOLOv1_ResNet_Multitasking_Joint_Detection_and_Segmentation(free_space%2Blane_marking)_BDD100K%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

~~Please download all the necessary files from the following link:~~

~~https://drive.google.com/file/d/1-KueKXnDbNMa06nurdZFQI7EmEsmQeiS/view?usp=drive_link~~

hw solution:
https://colab.research.google.com/drive/1MWC_tbjBPIh-6WiS-7RrVRm3EzneELDJ?usp=sharing








In [1]:
# update gdown for compatibility issue. 2022/12/12
!pip install --upgrade gdown

In [2]:
from google.colab import drive
import os

# The flag to determine where to put the downloaded files.
# True: zip file will be downloaded to this space
# False: manually download file folder from https://drive.google.com/file/d/16i4JoKP2Gyyry3yF0SzAjsdO8FZLXs4p/view?usp=sharing
# unzip it and upload the folder-YOLOv1_BDD100K_multitasking to the 1st level of your google drive which is actually "/content/drive/My Drive/"
In_this_space_flag = False

if In_this_space_flag == True and os.path.isdir("/content/YOLOv1_BDD100K_multitasking/") == False:
# 1 do everything in this temporary space
  #print('here')
  os.system("gdown https://drive.google.com/uc?id=16i4JoKP2Gyyry3yF0SzAjsdO8FZLXs4p")
  os.system("unzip YOLOv1_BDD100K_multitasking.zip")
  os.remove("YOLOv1_BDD100K_multitasking.zip")
elif In_this_space_flag == False: # and os.path.isdir("/content/drive/My Drive/YOLOv1_BDD100K_multitasking/") == True:
  drive.mount('/content/drive')
  assert os.path.isdir("/content/drive/My Drive/YOLOv1_BDD100K_multitasking/") == True


Mounted at /content/drive


In [3]:
if In_this_space_flag == True:
  ## version 1: download file
  # data folder
  multitasking_root_dir = "/content/YOLOv1_BDD100K_multitasking/"
else:
  ## version 2: download file
  # data folder
  multitasking_root_dir = "/content/drive/My Drive/YOLOv1_BDD100K_multitasking/"

In [4]:
import torch
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from torchvision import transforms
import time
import os
import shutil

classes = ["vehicle"]
bbox_num = 2
box_scale = 19


# training image list
train_image_object_detection_list = multitasking_root_dir + "bdd100k_clear_detection_sampled_files/train_list_yolo_1c.txt"
# tesing image list
eval_image_object_detection_list = multitasking_root_dir + "bdd100k_clear_detection_sampled_files/val_list_yolo_1c.txt"

train_image_semantic_segmentation_list = multitasking_root_dir + "bdd100k_segmentation_sampled_files/train_list.txt"
eval_image_semantic_segmentation_list =  multitasking_root_dir + "bdd100k_segmentation_sampled_files/val_list.txt"

train_image_lane_marking_semantic_segmentation_list = multitasking_root_dir + "bdd100k_lane_marking_segmentation_sampled_files/train_list.txt"
eval_image_lane_marking_semantic_segmentation_list =  multitasking_root_dir + "bdd100k_lane_marking_segmentation_sampled_files/val_list.txt"

# folder for training or trained models
model_path = multitasking_root_dir + "trained_YOLO_model_50/"

# folder for detection result to be used for estimating mAP
detection_result_txt_folder = multitasking_root_dir + "/bdd100k_clear_detection_sampled_files/detection_temp_folder/"
# folder to store temp GT data (pkl files)
GT_temp_folder = multitasking_root_dir + "GT_temp/"

## the following xml path format and test list are required by mAP estimation
annopath = os.path.join(
    multitasking_root_dir,
    'bdd100k_clear_detection_sampled_files',
    'val',
    'annotations',
    '{:s}.xml')
# imagesetfile indicates where test.txt (testing list) is
imagesetfile = os.path.join(
    multitasking_root_dir,
    'bdd100k_clear_detection_sampled_files',
    'mAP_test_list.txt')

# segmentation result folder
det_result_test_folder = multitasking_root_dir + "/bdd100k_clear_detection_sampled_files/detection_temp_folder_resnet18-tree/"
seg_result_test_folder = multitasking_root_dir + "/bdd100k_segmentation_sampled_files/segmentation_temp_folder_resnet18-tree/"
lm_seg_result_test_folder = multitasking_root_dir + "/bdd100k_lane_marking_segmentation_sampled_files/lm_segmentation_temp_folder_resnet18-tree/"

# make folders in case they don't exist
os.makedirs(det_result_test_folder, exist_ok=True)
os.makedirs(seg_result_test_folder, exist_ok=True)
os.makedirs(lm_seg_result_test_folder, exist_ok=True)
os.makedirs(GT_temp_folder, exist_ok=True)
os.makedirs(GT_temp_folder, exist_ok=True)
os.makedirs(model_path, exist_ok=True)

# alex: assignment revision here
num_class_seg = 2
num_class_lm_seg = 4+1 # lane marking segmentation

# class 0 (others) is [0,0,0]
lm_seg_class_colors = np.array([
    [  0, 0, 0], # class 0 (background, not annotated in the original dataset)
    [  0, 255, 255],  # Class 1: single white
    [  0, 255,   0],  # Class 2: single yellow
    [255,   0, 255],  # Class 3: double white
    [255, 255,   0],  # Class 4: double yellow
], dtype=np.uint8)

# alex: assignment revision here
seg_class_colors = np.array([
    [  0, 0, 0], # class 0 (background, not annotated in the original dataset)
    [128, 64, 128],  # Class 1: free-space
], dtype=np.uint8)

# in case we switch dataset and forget to generate new GT temp file, always kill them
[os.remove(os.path.join(GT_temp_folder, f)) for f in os.listdir(GT_temp_folder) if os.path.isfile(os.path.join(GT_temp_folder, f))]
# log file for mAP estimation corresponding to each epoch
savelog_filename = multitasking_root_dir + "savelog.txt"

epochs_start = 1 #0
epochs_end = 50 #150
batch_size =  10 #48#32 # L4 # 32resnet-50, A100,48 #A100 resnet-18 and resnet-34, #12 #24(ResNet34, 2/4 for LM marking)#48 #24(rest34, T4) #12 resnet50 #12 #24 #24 originally # the size depends on the RAM of GPU
cls_num = len(classes)

## data augmentation
jitter = 0.2
hue = 0.1
saturation = 1.5
exposure = 1.5

# weightings of loss
l_coord = 5
l_noobj = 0.5

YOLO_cfg = {
    'ceils_size':19,
    'class_num':len(classes),
    'box_num':2,
    'image_size':[608,608]
}


In [5]:
# Model definition

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.autograd import Variable
import torch
import torch.nn as nn
from torchvision.models import resnet18, resnet34, resnet50

class ResNetBackbone(nn.Module):
    def __init__(self, backbone_name='resnet50', pretrained=True):
        super(ResNetBackbone, self).__init__()
        if backbone_name == 'resnet34':
            resnet = resnet34(pretrained=pretrained)
            self.out_channels = 512
        elif backbone_name == 'resnet50':
            resnet = resnet50(pretrained=pretrained)
            self.out_channels = 2048 # ResNet-50's last feature map channels
        elif backbone_name == 'resnet18':
            resnet = resnet18(pretrained=pretrained)
            self.out_channels = 512 # ResNet-18's last feature map channels
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        # Keep all layers up to the last convolutional layer
        self.features = nn.Sequential(*list(resnet.children())[:-2])  # Remove FC and AvgPool layers

    def forward(self, x):
        return self.features(x)

class YOLO(nn.Module):
    def __init__(self, cls_num, bbox_num=2, scale_size=7, pretrained=True, num_class_seg=2, num_class_lm_seg=8, backbone_name='resnet18'): # Add backbone_name parameter
        super(YOLO, self).__init__()
        self.cls_num = cls_num
        self.scale_size = scale_size
        self.bbox_num = bbox_num
        self.last_output = 5 * self.bbox_num + self.cls_num
        self.num_class_seg = num_class_seg
        self.num_class_lm_seg = num_class_lm_seg

        # Use the new ResNetBackbone class
        self.feature = ResNetBackbone(backbone_name=backbone_name, pretrained=pretrained)
        backbone_out_channels = self.feature.out_channels # Get output channels from backbone

        # Create the YOLO-specific layers, adjust input channels
        self.local_layer = nn.Sequential(
            conv_block(backbone_out_channels, 512, kernel_size=3, pool=False, stride=1),
            conv_block(512, 512, kernel_size=3, pool=False, stride=1),
            conv_block(512, 512, kernel_size=3, pool=False, stride=1),
            conv_block(512, self.last_output, kernel_size=3, pool=False, stride=1),
        )

        self.reg_layer = nn.Sequential(
            nn.Linear(backbone_out_channels * scale_size * scale_size, 4096), # Adjust input channels
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, self.last_output * scale_size * scale_size),
        )

        # Semantic segmentation layers with deconvolutions, adjust input channels
        self.segmentation_head = nn.Sequential(
            conv_block(backbone_out_channels, 512, kernel_size=3, pool=False, stride=1),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),  # Upsample by 2x
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # Upsample by 2x
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # Upsample by 2x
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # Upsample by 2x
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, self.num_class_seg, kernel_size=4, stride=2, padding=1),  # Upsample to 608x608
            # Removed Softmax here
        )

        # Semantic segmentation layers with deconvolutions, adjust input channels
        self.lm_segmentation_head = nn.Sequential(
            conv_block(backbone_out_channels, 512, kernel_size=3, pool=False, stride=1),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),  # Upsample by 2x
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # Upsample by 2x
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # Upsample by 2x
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # Upsample by 2x
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, self.num_class_lm_seg, kernel_size=4, stride=2, padding=1),  # Upsample to 608x608
            # Removed Softmax here
        )


        # Create anchors
        cx = torch.linspace(0.5 / scale_size, (scale_size - 0.5) / scale_size, steps=scale_size).view(-1, scale_size).repeat(scale_size, 1).view(scale_size, scale_size, -1)
        cy = torch.linspace(0.5 / scale_size, (scale_size - 0.5) / scale_size, steps=scale_size).view(scale_size, -1).repeat(1, scale_size).view(scale_size, scale_size, -1)
        self.anchor = torch.cat((cx, cy), 2)


    def forward(self, x):
        B = x.size(0)
        features = self.feature(x)
        output = self.local_layer(features)
        output = output.view(output.size(0), -1)
        #output = self.reg_layer(output)
        output = output.view(-1, self.bbox_num * 5 + self.cls_num, self.scale_size, self.scale_size)
        output = output.permute(0, 2, 3, 1).contiguous()

        # Split output into classification, bbox regression, and confidence
        pred_cls = output[:, :, :, :self.cls_num]
        pred_bbox = torch.cat([output[:, :, :, self.cls_num + 5 * j:self.cls_num + 4 + 5 * j] for j in range(self.bbox_num)], -1)
        pred_response = torch.cat([output[:, :, :, self.cls_num + 4 + 5 * j:self.cls_num + 4 + 5 * j + 1] for j in range(self.bbox_num)], -1)

        # Add anchors to bbox predictions
        anchors = self.anchor.repeat((B, 1, 1, 1)).to(pred_bbox.device)
        pred_bbox[:, :, :, 0:2] += anchors

        # Semantic segmentation branch
        segmentation_output = self.segmentation_head(features)

        lm_segmentation_output = self.lm_segmentation_head(features)

        return pred_cls, pred_response, pred_bbox, segmentation_output, lm_segmentation_output


# Helper function for convolutional blocks
def conv_block(in_channels, out_channels, kernel_size, pool=False, stride=1):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=kernel_size // 2, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.LeakyReLU(0.1, inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*layers)

In [6]:
##BBox encoder, decoder for the last layer (7*7*30) and Non-Maximal Suppression

def yolo_box_encoder(bs):

    bb_class = np.zeros((YOLO_cfg['ceils_size'], YOLO_cfg['ceils_size'], YOLO_cfg['class_num']))
    bb_response = np.zeros((YOLO_cfg['ceils_size'], YOLO_cfg['ceils_size'], YOLO_cfg['box_num']))
    bb_boxes = np.zeros((YOLO_cfg['ceils_size'], YOLO_cfg['ceils_size'], 4 * YOLO_cfg['box_num']))

    for i in range(bs.shape[0]):

        local_x = int( min(0.99,bs[i,0] + bs[i,2] / 2) * YOLO_cfg['ceils_size'])
        local_y = int( min(0.99,bs[i,1] + bs[i,3] / 2) * YOLO_cfg['ceils_size'])

        for j in range(YOLO_cfg['box_num']):
            bb_response[local_y, local_x, j] = 1

            bb_boxes[local_y, local_x, j * 4 + 0] = (bs[i,0] + bs[i,2] /2)
            bb_boxes[local_y, local_x, j * 4 + 1] = (bs[i,1] + bs[i,3] /2)
            bb_boxes[local_y, local_x, j * 4 + 2] = np.sqrt(bs[i,2])
            bb_boxes[local_y, local_x, j * 4 + 3] = np.sqrt(bs[i,3])

        #print(f"------------int(bs[i,4])={int(bs[i,4])}")
        bb_class[local_y, local_x, int(bs[i,4])] = 1

    bb_boxes = torch.from_numpy(bb_boxes).float()
    bb_class = torch.from_numpy(bb_class).float()
    bb_response = torch.from_numpy(bb_response).float()
    boxes = (bb_class,bb_response,bb_boxes)

    return boxes

def py_cpu_nms(dets, scores, thresh):
    # dets:(m,5)  thresh:scaler
    #print(scores.shape)
    temp_len = 0  # np.max(dets[:,2]) * 0.05

    x1 = dets[:, 0]
    y1 = dets[:, 1]
    x2 = x1 + dets[:, 2]  # dets[:, 2]#
    y2 = y1 + dets[:, 3]  # dets[:, 3]#

    areas = (y2 - y1 + temp_len) * (x2 - x1 + temp_len)

    keep = []

    index = scores.argsort()[::-1][:200]


    while index.size > 0:
        i = index[0]  # every time the first is the biggst, and add it directly
        keep.append(i)

        x11 = np.maximum(x1[i], x1[index[1:]])  # calculate the points of overlap
        y11 = np.maximum(y1[i], y1[index[1:]])
        x22 = np.minimum(x2[i], x2[index[1:]])
        y22 = np.minimum(y2[i], y2[index[1:]])

        w = np.maximum(0, x22 - x11 + temp_len)  # the weights of overlap
        h = np.maximum(0, y22 - y11 + temp_len)  # the height of overlap

        overlaps = w * h
        #assert overlaps.all() >= 0
        ious = overlaps / (areas[i] + areas[index[1:]] - overlaps)

        idx = np.where(ious <= thresh)[0]
        index = index[idx + 1]

    return keep

def yolo_box_decoder(pred,conf_thresh=0,nms_thresh=0.5): # conf_thresh=0.01
    box_scale = YOLO_cfg['ceils_size']
    cls_num = YOLO_cfg['class_num']
    box_num = YOLO_cfg['box_num']

    pred_cls, pred_response, pred_bboxes = pred

    pred_cls = pred_cls.cpu()
    pred_response = pred_response.cpu()
    pred_bboxes = pred_bboxes.cpu()

    prob = pred_response
    max_prob, max_prob_index = prob.max(3)
    boxes = []
    classes = []


    max_prob_index = max_prob_index.permute(1,2,0)
    for B in range(1):
        for cls in range(cls_num):

            cls_prob = (pred_cls[B,:,:,cls]*max_prob[B, :, :]).data

            mask_box = torch.zeros((box_scale,box_scale,box_num))

            mask_box.scatter_(2,max_prob_index,1)
            mask_box = mask_box.unsqueeze(-1)

            mask_box = mask_box.repeat(1,1,1,4).view(box_scale,box_scale,box_num*4)
            mask_box = mask_box.unsqueeze(0).byte()
            mask_box = mask_box.expand_as(pred_bboxes)
            bbox = pred_bboxes[mask_box.to(torch.bool)].data

            bbox = bbox.reshape(-1,4)
            cls_prob = cls_prob.reshape(-1,1)

            a = cls_prob.gt(conf_thresh)
            mask_a = a.expand_as(bbox)

            bbox = bbox[mask_a].reshape(-1,4)
            cls_prob = cls_prob[a].reshape(-1,1)

            if bbox.shape[0] > 0:
                bbox[:,0:2] = bbox[:,0:2] - 0.5*torch.pow(bbox[:,2:4],2)
                bbox[:,2:4] = torch.pow(bbox[:,2:4],2)

                pre_cls_box = bbox.data.numpy()
                pre_cls_score = cls_prob.data.view(-1).numpy()

                keep = py_cpu_nms(pre_cls_box, pre_cls_score, thresh=nms_thresh)
                for conf_keep, loc_keep in zip(pre_cls_score[keep], pre_cls_box[keep]):
                    boxes.append(loc_keep)
                    classes.append([cls,conf_keep])

    boxes = np.array(boxes)
    classes = np.array(classes)

    return boxes,classes



In [7]:
# data loader and data augmentation

from PIL import Image,ImageEnhance
from torch.utils import data
import random

def distort_image(im, hue, sat, val):
    im = im.convert('HSV')
    cs = list(im.split())
    # cs[1] = cs[1].point(lambda i: i * sat)
    # cs[2] = cs[2].point(lambda i: i * val)

    # def change_hue(x):
    #     x += hue*255
    #     if x > 255:
    #         x -= 255
    #     if x < 0:
    #         x += 255
    #     return x
    cs[1] = cs[1].point(lambda i: int(float(i) * sat))
    cs[2] = cs[2].point(lambda i: int(float(i) * val))

    def change_hue(x):
        x += hue*float(255)
        if x > 255:
            x -= 255
        if x < 0:
            x += 255
        return int(x)
    cs[0] = cs[0].point(change_hue)
    im = Image.merge(im.mode, tuple(cs))

    im = im.convert('RGB')
    return im

def rand_scale(s):
    scale = random.uniform(1, s)
    if(random.randint(1,10000)%2):
        return scale
    return 1./scale

def random_distort_image(im, hue, saturation, exposure):
    dhue = random.uniform(-hue, hue)
    dsat = rand_scale(saturation)
    dexp = rand_scale(exposure)

    res = distort_image(im, dhue, dsat, dexp)
    return res


def data_augmentation(img, shape, jitter, hue, saturation, exposure):
    oh = img.height
    ow = img.width

    dw =int(ow*jitter)
    dh =int(oh*jitter)

    pleft  = random.randint(-dw, dw)
    pright = random.randint(-dw, dw)
    ptop   = random.randint(-dh, dh)
    pbot   = random.randint(-dh,dh)


    swidth =  ow - pleft - pright
    sheight = oh - ptop - pbot


    sx = float(swidth)  / ow
    sy = float(sheight) / oh

    flip = random.randint(1,10000)%2

    cropped = img.crop( (pleft, ptop, pleft + swidth - 1, ptop + sheight - 1))

    dx = (float(pleft)/ow)/sx
    dy = (float(ptop) /oh)/sy


    sized = cropped.resize(shape,Image.NEAREST)
    if flip:
        sized = sized.transpose(Image.FLIP_LEFT_RIGHT)
    img = random_distort_image(sized, hue, saturation, exposure)

    return img, flip, dx,dy,sx,sy

def fill_truth_detection(bs, flip, dx, dy, sx, sy):

    new_bs = []
    #print(bs)
    for i in range(bs.shape[0]):
        x1 = bs[i][0]
        y1 = bs[i][1]
        x2 = bs[i][0] + bs[i][2]
        y2 = bs[i][1] + bs[i][3]

        x1 = min(0.99, max(0, x1 * sx - dx))
        y1 = min(0.99, max(0, y1 * sy - dy))

        x2 = max(0, min(0.999, x2 * sx - dx))
        y2 = max(0, min(0.999, y2 * sy - dy))


        bs[i][0] = x1
        bs[i][1] = y1
        bs[i][2] = x2 - x1
        bs[i][3] = y2 - y1
        bs[i][4] = bs[i][4]
        if flip:
            bs[i][0] =  1 - bs[i][0] - bs[i][2]

        if bs[i][2] > 0 and bs[i][3] > 0:
            new_bs.append([bs[i]])

    new_bs = np.array(new_bs)
    new_bs = np.reshape(new_bs, (-1, 5))

    return new_bs


def norm_bb(b,size):
    x = b[:, 0:1]
    y = b[:, 1:2]

    dw = 1. / size[0]
    dh = 1. / size[1]

    x = (x * dw).clip(0.01, 0.99)
    y = (y * dh).clip(0.01, 0.99)
    w = ((b[:, 2:3] - b[:, 0:1]) * dw).clip(0.01, 0.99)
    h = ((b[:, 3:4] - b[:, 1:2]) * dh).clip(0.01, 0.99)

    if np.all(b[:, 4:5]>1):
      raise(f"----------b[:, 4:5]={np.all(b[:, 4:5])} but it shouldn't be")
    return np.concatenate((x, y, w, h, b[:, 4:5]), axis=1)

def load_data_detection(imgpath, labpath ,shape, aug = True, jitter=0.2, hue=0.1, saturation=1.5, exposure=1.5):


    img = Image.open(imgpath).convert('RGB')

    bs = np.loadtxt(labpath,delimiter=',')
    bs = np.reshape(bs, (-1, 5))

    bs = norm_bb(bs,(img.width,img.height))

    # data augmentation of YOLOv1 doesn't fit ResNet34 version because of the input range is not between 0-255
    #if aug:
    #    img, flip, dx, dy, sx, sy = data_augmentation(img, shape, jitter, hue, saturation, exposure)
    #else:
    flip, dx, dy, sx, sy = False, 0, 0, 1, 1

    #print(dx, dy, sx, sy)
    label = fill_truth_detection(bs, flip, dx, dy, 1./sx, 1./sy)

    return img,label

def load_data_semantic_segmentation(imgpath, labpath, shape, num_classes, seg_class_colors):
    """
    Load input image and preprocess segmentation label for binary classification.

    Args:
        imgpath (str): Path to the input image.
        labpath (str): Path to the segmentation label image.
        shape (tuple): Target shape (width, height).
        num_classes (int): Number of segmentation classes.

    Returns:
        img (PIL.Image): Resized input image.
        binary_label (np.ndarray): Binary label with shape (H, W).
        one_hot_label (np.ndarray): One-hot encoded label with shape (num_classes, H, W).
    """
    # Load and resize the input image
    img = Image.open(imgpath).convert('RGB')
    img = img.resize(shape, resample=Image.NEAREST)

    # Load and resize the label image
    label_img = Image.open(labpath).convert('RGB')
    label_img = label_img.resize(shape, resample=Image.NEAREST)

    # Convert label image to numpy array
    label = np.asarray(label_img)

    # alex: assignment revision here.
    # The goal is to create a new tensor-class_label of shape H*W where each element value is either 0,1,2
    # because we would like to have 3 classes. please make sure you know the difference between
    # label == seg_class_colors[1] and
    # np.all(label == seg_class_colors[1], axis=-1)
    # Create binary label: class 1 for [128, 64, 128], else class 0
    binary_label = np.all(label == seg_class_colors[1], axis=-1).astype(np.uint8)

    # Ensure binary_label is integer type (required for one-hot encoding)
    binary_label = binary_label.astype(np.int64)

    # Convert binary label to one-hot encoding
    one_hot_label = np.eye(num_classes, dtype=np.uint8)[binary_label]  # Shape: (H, W, num_classes)

    # Rearrange one-hot to (num_classes, H, W) for PyTorch compatibility
    one_hot_label = np.transpose(one_hot_label, (2, 0, 1))

    return img, binary_label, one_hot_label

def load_data_lm_semantic_segmentation(imgpath, labpath, shape, num_classes, lm_seg_class_colors):
    """
    Load input image and preprocess segmentation label for multi-class classification.

    Args:
        imgpath (str): Path to the input image.
        labpath (str): Path to the segmentation label image.
        shape (tuple): Target shape (width, height).
        num_classes (int): Number of segmentation classes.
        lm_seg_class_colors (np.ndarray): Array of RGB colors for each class.

    Returns:
        img (PIL.Image): Resized input image.
        class_label (np.ndarray): Class index label with shape (H, W).
        one_hot_label (np.ndarray): One-hot encoded label with shape (num_classes, H, W).
    """

    #print(f"---lm_seg_class_colors={lm_seg_class_colors }")
    import numpy as np
    from PIL import Image

    # Load and resize input image
    img = Image.open(imgpath).convert('RGB')
    img = img.resize(shape, resample=Image.NEAREST)

    # Load and resize label image
    label_img = Image.open(labpath).convert('RGB')
    label_img = label_img.resize(shape, resample=Image.NEAREST)
    label = np.asarray(label_img)

    # Initialize class label map with zeros (default class index 0 for background)
    class_label = np.zeros((label.shape[0], label.shape[1]), dtype=np.uint8)

    # Map each color to its class index (0–4) where 0 is BG.
    # We start from index 1 in lm_seg_class_colors because index 0 is the background color [0,0,0]
    # which is handled by the initialization of class_label.
    for idx, color in enumerate(lm_seg_class_colors[1:]):
        mask = np.all(label == color, axis=-1)
        # Assign the correct class index (idx + 1 because we skipped index 0)
        class_label[mask] = idx + 1

    # Ensure class_label is integer type (required for one-hot encoding)
    class_label = class_label.astype(np.int64)


    # Convert to one-hot encoding
    one_hot_label = np.eye(num_classes, dtype=np.uint8)[class_label]  # Shape: (H, W, num_classes)
    one_hot_label = np.transpose(one_hot_label, (2, 0, 1))  # Shape: (num_classes, H, W)

    return img, class_label, one_hot_label

class VOCDatasets(data.Dataset):
    def __init__(self, transform, object_detection_list_file, semantic_segmentation_list_file, lane_marking_semantic_segmentation_list_file, multitasking_root_dir, num_class_seg, num_class_lm_seg, seg_class_colors, lm_seg_class_colors, box_encoder = None, train=False):

        self.transform = transform
        self.train = train
        self.object_detection_image_path = []
        self.object_detection_label_path = []

        self.semantic_segmentation_image_path = []
        self.semantic_segmentation_label_path = []

        self.lane_marking_semantic_segmentation_image_path = []
        self.lane_marking_semantic_segmentation_label_path = []

        self.box_encoder = box_encoder

        self.num_class_seg = num_class_seg
        self.num_class_lm_seg = num_class_lm_seg
        self.seg_class_colors = seg_class_colors
        self.lm_seg_class_colors = lm_seg_class_colors

        # od file-reading
        with open(object_detection_list_file) as f:
            object_detection_lines = f.readlines()

        # ss file-reading
        with open(semantic_segmentation_list_file) as f:
            semantic_segmentation_lines = f.readlines()

        # lane marking segmentation file-reading
        with open(lane_marking_semantic_segmentation_list_file) as f:
            lane_marking_semantic_segmentation_lines = f.readlines()

        self.object_detection_num_samples = len(object_detection_lines)
        self.semantic_segmentation_num_samples = len(semantic_segmentation_lines)
        self.lane_marking_semantic_segmentation_num_samples = len(lane_marking_semantic_segmentation_lines)

        self.num_samples = min(self.object_detection_num_samples, self.semantic_segmentation_num_samples, self.lane_marking_semantic_segmentation_num_samples)
        #self.num_samples = 24

        for line in object_detection_lines:
            # using "__" instead of "__" because the data path under Google drive is all "/content/drive/MyDrive/...
            # Therefore, the following .split(' ') function would result in error
            # splited = line.strip().split(' ')
            splited = line.strip().split('__')
            self.object_detection_image_path.append(multitasking_root_dir + splited[0])
            self.object_detection_label_path.append(multitasking_root_dir + splited[1])

        for line in semantic_segmentation_lines:
            # using "__" instead of "__" because the data path under Google drive is all "/content/drive/MyDrive/...
            # Therefore, the following .split(' ') function would result in error
            # splited = line.strip().split(' ')
            splited = line.strip().split('__')
            self.semantic_segmentation_image_path.append(multitasking_root_dir + splited[0])
            self.semantic_segmentation_label_path.append(multitasking_root_dir + splited[1])

        for line in lane_marking_semantic_segmentation_lines:
            # using "__" instead of "__" because the data path under Google drive is all "/content/drive/MyDrive/...
            # Therefore, the following .split(' ') function would result in error
            # splited = line.strip().split(' ')
            splited = line.strip().split('__')
            self.lane_marking_semantic_segmentation_image_path.append(multitasking_root_dir + splited[0])
            self.lane_marking_semantic_segmentation_label_path.append(multitasking_root_dir + splited[1])

    def __getitem__(self, idx):

        file_name_od = self.object_detection_image_path[idx]
        gt_path_od = self.object_detection_label_path[idx]

        file_name_ss = self.semantic_segmentation_image_path[idx]
        gt_path_ss = self.semantic_segmentation_label_path[idx]

        file_name_lm_ss = self.lane_marking_semantic_segmentation_image_path[idx]
        gt_path_lm_ss = self.lane_marking_semantic_segmentation_label_path[idx]

        #img,bbox = data_augment(file_name,gt_path,self.train)
        #print('--in __getitem__, file_name={}'.format(file_name))
        img_od, bbox = load_data_detection(file_name_od, gt_path_od, [608, 608], self.train)

        img_ss, label_ss, one_hot_label_ss =              load_data_semantic_segmentation(file_name_ss, gt_path_ss, [608, 608], self.num_class_seg, self.seg_class_colors)
        img_lm_ss, label_lm_ss, one_hot_lm_label_ss = load_data_lm_semantic_segmentation(file_name_lm_ss, gt_path_lm_ss, [608, 608], self.num_class_lm_seg, self.lm_seg_class_colors)

        #maximum bbox=50

        if self.box_encoder is not None:
            gt = self.box_encoder(bbox)
        else:
            gt = np.zeros((len(bbox), 5), dtype=np.float32)
            gt[:len(bbox), :] = bbox
            #print(gt.shape)
            gt = torch.from_numpy(gt).float()

        img_od = self.transform(img_od)
        img_ss = self.transform(img_ss)
        img_lm_ss = self.transform(img_lm_ss)

        label_ss = torch.from_numpy(label_ss.copy()).long()
        label_lm_ss = torch.from_numpy(label_lm_ss.copy()).long()

        one_hot_label_ss = torch.from_numpy(one_hot_label_ss.copy()).float()
        one_hot_label_lm_ss = torch.from_numpy(one_hot_lm_label_ss.copy()).float()

        return img_od, gt, img_ss, label_ss, one_hot_label_ss, img_lm_ss, label_lm_ss, one_hot_label_lm_ss

    def __len__(self):
        return self.num_samples

# remember that the following tansform would only accept PIL images as input
transform = transforms.Compose([
    transforms.Resize([608, 608]),
    transforms.ToTensor(),
    #transforms.Normalize([0.485, 0.456, 0.406], [1, 1, 1])
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize with ImageNet stats
])


dataset = VOCDatasets(transform, train_image_object_detection_list, train_image_semantic_segmentation_list, train_image_lane_marking_semantic_segmentation_list, multitasking_root_dir, num_class_seg, num_class_lm_seg, seg_class_colors, lm_seg_class_colors, yolo_box_encoder, train=True) # train=true will apply data augmentation

train_loader = data.DataLoader(dataset=dataset,
                              batch_size=batch_size,
                              shuffle=True,
                              num_workers=8)

In [8]:
# YOLO loss

import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.autograd import Variable

class yolov1_loss(nn.Module):
    def __init__(self, B, l_coord, l_noobj,device='cuda' ,cls_num=20):
        super(yolov1_loss, self).__init__()
        self.B = B # bbox_num
        self.l_coord = l_coord
        self.l_noobj = l_noobj
        self.class_num = cls_num
        self.device = device

    def compute_iou(self, box1, box2):
        '''Compute the intersection over union of two set of boxes, each box is [x1,y1,x2,y2].
        Args:
          box1: (tensor) bounding boxes, sized [N,4].
          box2: (tensor) bounding boxes, sized [M,4].
        Return:
          (tensor) iou, sized [N,M].
        '''

        lt = torch.max(
            box1[:, :2],  # [N,2] -> [N,1,2] -> [N,M,2]
            box2[:, :2],  # [M,2] -> [1,M,2] -> [N,M,2]
        )

        rb = torch.min(
            box1[:, 2:],  # [N,2] -> [N,1,2] -> [N,M,2]
            box2[:, 2:],  # [M,2] -> [1,M,2] -> [N,M,2]
        )

        wh = rb - lt  # [N,M,2]
        wh[wh < 0] = 0  # clip at 0
        inter = wh[:, 0] * wh[:, 1]  # [N,M]

        area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])  # [N,]
        area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])  # [M,]

        iou = inter / (area1 + area2 - inter)
        return iou

    def forward(self,pred,target):
        #print(target)
        pred_cls, pred_response, pred_bboxes = pred
        label_cls, label_response, label_bboxes = target


        pred_cls = pred_cls.to(self.device)
        pred_response =  pred_response.to(self.device)
        pred_bboxes = pred_bboxes.to(self.device)

        label_cls = label_cls.to(self.device)
        label_response =  label_response.to(self.device)
        label_bboxes = label_bboxes.to(self.device)


        batch_size = pred_response.size(0)

        no_obj_mask = (label_response[:, :, :, 0] < 1).unsqueeze(-1).expand_as(label_response)

        obj_response_mask = (label_response[:, :, :, 0] > 0).unsqueeze(-1).expand_as(label_response)

        obj_box_mask = (label_response[:, :, :, 0] > 0).unsqueeze(-1).expand_as(label_bboxes)

        obj_cls_mask = (label_response[:, :, :, 0] > 0).unsqueeze(-1).expand_as(label_cls)

        no_obj_contain_pred = pred_response[no_obj_mask].view(-1)
        no_obj_contain_target = label_response[no_obj_mask].view(-1)


        obj_contain_pred = pred_response[obj_response_mask].view(-1, self.B)

        obj_contain_target = label_response[obj_response_mask].view(-1, self.B)

        # class pred response
        obj_class_pred = pred_cls[obj_cls_mask].view(-1, self.class_num)
        obj_class_target = label_cls[obj_cls_mask].view(-1, self.class_num)

        # box pred response
        obj_loc_pred = pred_bboxes[obj_box_mask].view(-1, self.B * 4)
        obj_loc_target = label_bboxes[obj_box_mask].view(-1, self.B * 4)

        iou = torch.zeros(obj_loc_pred.size(0), self.B)
        iou = Variable(iou)

        for j in range(self.B):
            pred_bb = torch.zeros(obj_loc_pred.size(0), 4)
            pred_bb = Variable(pred_bb)

            target_bb = torch.zeros(obj_loc_pred.size(0), 4)
            target_bb = Variable(target_bb)

            target_bb[:, 0] = obj_loc_target[:, j * 4] - 0.5 * pow(obj_loc_target[:, j * 4 + 2], 2)
            target_bb[:, 1] = obj_loc_target[:, j * 4 + 1] - 0.5 * pow(obj_loc_target[:, j * 4 + 3], 2)
            target_bb[:, 2] = obj_loc_target[:, j * 4] + 0.5 * pow(obj_loc_target[:, j * 4 + 2], 2)
            target_bb[:, 3] = obj_loc_target[:, j * 4 + 1] + 0.5 * pow(obj_loc_target[:, j * 4 + 3], 2)

            pred_bb[:, 0] = obj_loc_pred[:, j * 4] - 0.5 * pow(obj_loc_pred[:, j * 4 + 2], 2)
            pred_bb[:, 1] = obj_loc_pred[:, j * 4 + 1] - 0.5 * pow(obj_loc_pred[:, j * 4 + 3], 2)
            pred_bb[:, 2] = obj_loc_pred[:, j * 4] + 0.5 * pow(obj_loc_pred[:, j * 4 + 2], 2)
            pred_bb[:, 3] = obj_loc_pred[:, j * 4 + 1] + 0.5 * pow(obj_loc_pred[:, j * 4 + 3], 2)

            iou[:, j] = self.compute_iou(target_bb, pred_bb)

        max_iou, max_index = iou.max(1)
        min_iou, _ = iou.min(1)
        max_index = max_index.data.cpu()

        coo_response_mask = torch.ByteTensor(obj_loc_pred.size(0), self.B * 4).to(self.device)

        coo_response_mask.zero_()
        for i in range(obj_loc_pred.size(0)):
            coo_response_mask[i, max_index[i] * 4:max_index[i] * 4 + 4] = 1

        obj_axis_pred = obj_loc_pred[coo_response_mask.to(torch.bool)].view(-1, 4)
        obj_axis_target = obj_loc_target[coo_response_mask.to(torch.bool)].view(-1, 4)

        iou_response_mask = coo_response_mask[:, [i * 4 for i in range(self.B)]]

        obj_response_pred = obj_contain_pred[iou_response_mask.to(torch.bool)].view(-1)
        obj_response_target = obj_contain_target[iou_response_mask.to(torch.bool)].view(-1)

        obj_local_loss = F.mse_loss(obj_axis_pred[:, 0:2], obj_axis_target[:, 0:2], size_average=False) + \
                         F.mse_loss(obj_axis_pred[:, 2:4], obj_axis_target[:, 2:4], size_average=False)
        obj_class_loss = F.mse_loss(obj_class_pred, obj_class_target, size_average=False)


        max_iou = (max_iou.data).to(self.device)
        conf_id = ((1 - max_iou) * self.l_noobj + max_iou).to(self.device)

        conf_id = Variable(conf_id, requires_grad=True)

        obj_contain_loss = F.mse_loss(obj_response_pred, max_iou, size_average=False)

        no_obj_contain_loss = F.mse_loss(no_obj_contain_pred, no_obj_contain_target, size_average=False)

        iou_loss = F.mse_loss(max_iou, obj_response_target, size_average=False)

        loss_all = (self.l_coord * obj_local_loss + obj_class_loss + obj_contain_loss + self.l_noobj * no_obj_contain_loss + iou_loss) / batch_size

        loss_info = {
            'local_loss': self.l_coord * obj_local_loss.data,
            'class_loss': obj_class_loss.data,
            'contain_loss': obj_contain_loss.data,
            'no_contain_loss': self.l_noobj * no_obj_contain_loss,
            'iou_loss': iou_loss,
            'mean_iou': torch.mean(max_iou)
        }

        return loss_all, loss_info

In [9]:
# optimizer and loss


device='cuda'

loss_detect = yolov1_loss(bbox_num, l_coord, l_noobj, device, cls_num)

# do remind the difference between step & epoch
step = epochs_start * len(train_loader)

best_score = 0
eval_score = 0

In [10]:
# mAP estimation


import xml.etree.ElementTree as ET
import os
import pickle
import numpy as np
import cv2

#### segmentation metrics
# borrow functions and modify it from https://github.com/Kaixhin/FCN-semantic-segmentation/blob/master/main.py
# Calculates class intersections over unions
def iou(pred, target, num_class):
    ious = []
    for cls in range(num_class):
        pred_inds = pred == cls
        target_inds = target == cls
        intersection = pred_inds[target_inds].sum()
        union = pred_inds.sum() + target_inds.sum() - intersection
        if union == 0:
            ious.append(float('nan'))  # if there is no ground truth, do not include in evaluation
        else:
            ious.append(float(intersection) / max(union, 1))
        # print("cls", cls, pred_inds.sum(), target_inds.sum(), intersection, float(intersection) / max(union, 1))
    return ious


def pixel_acc(pred, target):
    correct = (pred == target).sum()
    total   = (target == target).sum()
    return correct / total


def parse_rec(filename):
    """ Parse a PASCAL VOC xml file """
    tree = ET.parse(filename)
    objects = []
    for obj in tree.findall('object'):
        obj_struct = {}
        obj_struct['name'] = obj.find('name').text
        #obj_struct['pose'] = obj.find('pose').text
        obj_struct['truncated'] = int(obj.find('truncated').text)
        #obj_struct['difficult'] = int(obj.find('difficult').text)
        obj_struct['difficult'] = int(0)
        bbox = obj.find('bndbox')
        obj_struct['bbox'] = [int(bbox.find('xmin').text),
                              int(bbox.find('ymin').text),
                              int(bbox.find('xmax').text),
                              int(bbox.find('ymax').text)]
        objects.append(obj_struct)

    return objects


def voc_ap(rec, prec, use_07_metric=False):
    """ ap = voc_ap(rec, prec, [use_07_metric])
    Compute VOC AP given precision and recall.
    If use_07_metric is true, uses the
    VOC 07 11 point method (default:False).
    """
    if use_07_metric:
        # 11 point metric
        ap = 0.
        for t in np.arange(0., 1.1, 0.1):
            if np.sum(rec >= t) == 0:
                p = 0
            else:
                p = np.max(prec[rec >= t])
            ap = ap + p / 11.
    else:
        # correct AP calculation
        # first append sentinel values at the end
        mrec = np.concatenate(([0.], rec, [1.]))
        mpre = np.concatenate(([0.], prec, [0.]))

        # compute the precision envelope
        for i in range(mpre.size - 1, 0, -1):
            mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])

        # to calculate area under PR curve, look for points
        # where X axis (recall) changes value
        i = np.where(mrec[1:] != mrec[:-1])[0]

        # and sum (\Delta recall) * prec
        ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    return ap


def voc_eval(detpath,
             annopath,
             imagesetfile,
             classname,
             cachedir,
             ovthresh=0.5,
             use_07_metric=False):
    """rec, prec, ap = voc_eval(detpath,
                                annopath,
                                imagesetfile,
                                classname,
                                [ovthresh],
                                [use_07_metric])

    Top level function that does the PASCAL VOC evaluation.

    detpath: Path to detections
        detpath.format(classname) should produce the detection results file.
    annopath: Path to annotations
        annopath.format(imagename) should be the xml annotations file.
    imagesetfile: Text file containing the list of images, one image per line.
    classname: Category name (duh)
    cachedir: Directory for caching the annotations
    [ovthresh]: Overlap threshold (default = 0.5)
    [use_07_metric]: Whether to use VOC07's 11 point AP computation
        (default False)
    """
    # assumes detections are in detpath.format(classname)
    # assumes annotations are in annopath.format(imagename)
    # assumes imagesetfile is a text file with each line an image name
    # cachedir caches the annotations in a pickle file

    # first load gt
    if not os.path.isdir(cachedir):
        os.mkdir(cachedir)
    cachefile = os.path.join(cachedir, 'annots.pkl')
    # read list of images
    with open(imagesetfile, 'r') as f:
        lines = f.readlines()
    imagenames = [x.strip() for x in lines]


    if not os.path.isfile(cachefile):
        # load annots
        recs = {}
        for i, imagename in enumerate(imagenames):
            recs[imagename] = parse_rec(annopath.format(imagename))
            if i % 100 == 0:
                print('Reading annotation for {:d}/{:d}'.format(
                    i + 1, len(imagenames)))
        # save
        print('Saving cached annotations to {:s}'.format(cachefile))
        with open(cachefile, 'wb') as f:
            pickle.dump(recs, f)
    else:
        # load
        with open(cachefile, 'rb') as f:
            recs = pickle.load(f)
            #print('Using cached annotations from {:s}'.format(cachefile))

    #print(f'------------------recs={recs}')
    # extract gt objects for this class
    class_recs = {}
    npos = 0
    for imagename in imagenames:
        R = [obj for obj in recs[imagename] if obj['name'] == classname]
        bbox = np.array([x['bbox'] for x in R])
        difficult = np.array([x['difficult'] for x in R]).astype(np.bool_)
        det = [False] * len(R)
        npos = npos + sum(~difficult)
        class_recs[imagename] = {'bbox': bbox,
                                 'difficult': difficult,
                                 'det': det}

    # read dets
    detfile = detpath.format(classname)
    with open(detfile, 'r') as f:
        lines = f.readlines()

    splitlines = [x.strip().split(' ') for x in lines]
    image_ids = [x[0] for x in splitlines]
    confidence = np.array([float(x[1]) for x in splitlines])
    BB = np.array([[float(z) for z in x[2:]] for x in splitlines])

    # sort by confidence
    sorted_ind = np.argsort(-confidence)
    sorted_scores = np.sort(-confidence)

    BB = BB[sorted_ind, :]
    image_ids = [image_ids[x] for x in sorted_ind]

    # go down dets and mark TPs and FPs
    nd = len(image_ids)
    tp = np.zeros(nd)
    fp = np.zeros(nd)
    for d in range(nd):
        R = class_recs[image_ids[d]]
        bb = BB[d, :].astype(float)
        ovmax = -np.inf
        BBGT = R['bbox'].astype(float)

        if BBGT.size > 0:
            # compute overlaps
            # intersection
            ixmin = np.maximum(BBGT[:, 0], bb[0])
            iymin = np.maximum(BBGT[:, 1], bb[1])
            ixmax = np.minimum(BBGT[:, 2], bb[2])
            iymax = np.minimum(BBGT[:, 3], bb[3])
            iw = np.maximum(ixmax - ixmin + 1., 0.)
            ih = np.maximum(iymax - iymin + 1., 0.)
            inters = iw * ih

            # union
            uni = ((bb[2] - bb[0] + 1.) * (bb[3] - bb[1] + 1.) +
                   (BBGT[:, 2] - BBGT[:, 0] + 1.) *
                   (BBGT[:, 3] - BBGT[:, 1] + 1.) - inters)

            overlaps = inters / uni
            ovmax = np.max(overlaps)
            jmax = np.argmax(overlaps)

        if ovmax > ovthresh:
            if not R['difficult'][jmax]:
                if not R['det'][jmax]:
                    tp[d] = 1.
                    R['det'][jmax] = 1
                else:
                    fp[d] = 1.
        else:
            fp[d] = 1.

    # compute precision recall
    fp = np.cumsum(fp)
    tp = np.cumsum(tp)
    rec = tp / float(npos)
    # avoid divide by zero in case the first detection matches a difficult
    # ground truth
    prec = tp / np.maximum(tp + fp, np.finfo(np.float64).eps)
    ap = voc_ap(rec, prec, use_07_metric)

    return rec, prec, ap

def _do_python_eval_quite(res_prefix, annopath, imagesetfile, output_dir=''):
    # do remind only PAScal VOC 07 test needs to use use_07_metric ==  True
    # simply replace 2007 with year bigger than 2010 will work
    _year = '2007'

    _classes = ('__background__',  # always index 0
                'vehicle')
    filename = res_prefix + '{:s}.txt'

    # cachedir indicates where annots.pkl is!
    cachedir = GT_temp_folder
    aps = []

    result = dict()
    # The PASCAL VOC metric changed in 2010
    use_07_metric = True if int(_year) < 2010 else False
    # print('VOC07 metric? ' + ('Yes' if use_07_metric else 'No'))
    if not os.path.isdir(output_dir):
        os.mkdir(output_dir)

    print("Begin to perform mAP estimation")
    for i, cls in enumerate(_classes):
        if cls == '__background__':
            continue

        rec, prec, ap = voc_eval(
            filename, annopath, imagesetfile, cls, cachedir, ovthresh=0.5,
            use_07_metric=use_07_metric)
        aps += [ap]
        print('AP for {} = {:.4f}'.format(cls, ap))
        result[cls] = ap
        # the following file (rec, prec, & ap of each class) is only for back-up.
        with open(os.path.join(output_dir, cls + '_pr.pkl'), 'wb') as f:
            pickle.dump({'rec': rec, 'prec': prec, 'ap': ap}, f)

    return result


def test_result_od(epoch, model, test_loader, test_list_od, prefix, multitasking_root_dir, outfile, det_result_test_folder):

    classes = ['vehicle']
    class_num = cls_num


    lines = []
    with open(test_list_od) as f:
        lines = f.readlines()

    fps = [0] * class_num
    if not os.path.exists(outfile):
        os.mkdir(outfile)

    for i in range(class_num):
        buf = '%s/%s%s.txt' % (prefix, outfile, classes[i])
        fps[i] = open(buf, 'w')

    top_imgs_to_be_saved = 8
    image_list = []

    # do remember that label (of testing data) is not used in the following for-loop
    # the detection results stored in txt files will be evaluated in _do_python_eval_quite() by reading xml files.
    #img_od, target, img_ss, label_ss, one_hot_label_ss
    for lineId, (img_od, label_od, img_ss, label_ss, one_hot_label_ss, _, _, _) in enumerate(test_loader):
        t1 = time.time()
        #img = cv2.imread(lines[lineId].split(" ")[0])

        # fetch the original width and height
        img = cv2.imread(multitasking_root_dir + lines[lineId].split("__")[0])
        width, height = img.shape[1], img.shape[0]

        img_od = img_od.cuda()

        t2 = time.time()
        #print("load times: ",t2-t1)

        t1 = time.time()

        pred_cls, pred_response, pred_bboxes, _, _ = model(img_od)

        pred = pred_cls, pred_response, pred_bboxes


        t2 = time.time()
        #print("pred times: ",t2-t1)


        t1 = time.time()
        pred_boxes,pred_conf = yolo_box_decoder(pred)
        t2 = time.time()
        #print(f"-------pred_conf={pred_conf}")
        #print("decoder times: ",t2-t1)

        fileId = os.path.basename(lines[lineId]).split('.')[0]

        for j in range(len(pred_boxes)):

            x1 = pred_boxes[j,0]
            y1 = pred_boxes[j,1]
            x2 = x1 + pred_boxes[j,2]
            y2 = y1 + pred_boxes[j,3]

            x1,x2 = x1*width,x2*width
            y1,y2 = y1*height,y2*height

            cls_id = int(pred_conf[j,0])
            scores = pred_conf[j,1]
            #print(scores)
            fps[cls_id].write('%s %.3f %.1f %.1f %.1f %.1f\n' %
                              (fileId, scores, x1 + 1, y1 + 1, x2 + 1, y2 + 1))

            ## uncomment the following section to produce detection results of the current epoch

            if scores > 0.2:
                cv2.rectangle(img,(int(x1),int(y1)-20),(int(x2),int(y1)),(125,125,125),-1)
                img = cv2.putText(img, classes[cls_id] + ': %.2f' % round(scores,3), (int(x1),int(y1)),  cv2.FONT_HERSHEY_PLAIN, 1.5, (0,0,0), thickness=1)
                #img = cv2.putText(img, str(round(scores,3)), (int(x1),int(y1)+20),  cv2.FONT_HERSHEY_PLAIN, 2.0, (0,0,0), thickness=1)
                img = cv2.rectangle(img, (int(x1),int(y1)), (int(x2),int(y2)), (0, 255, 0), 2)
                # filename = lines[lineId].split(" ")[0]
                # filename_only = filename.split("/")[-1]
                # cv2.imwrite('/content/drive/MyDrive/YOLO_data_model/voc07_det_results/' + filename_only, img)

        image_list.append(img)

        if lineId < top_imgs_to_be_saved:
            concatenated_image = cv2.vconcat(image_list)
            file_name = f"{det_result_test_folder}{epoch}_batched.jpg"
            cv2.imwrite(file_name, concatenated_image)


    for i in range(class_num):
        fps[i].close()


def test_result_ss(epoch, model, test_loader, seg_result_test_folder, num_class_seg, seg_class_colors):

    total_ious = []
    pixel_accs = []
    top_imgs_to_be_saved = 8

    # Lists to store images, predictions, and ground truth for batch visualization
    batched_img_ss = []
    batched_pred_seg = []
    batched_label_ss = []

    for lineId, (img_od, label_od, img_ss, label_ss, one_hot_label_ss, _, _, _) in enumerate(test_loader):

        img_ss = img_ss.cuda()

        _, _, _, pred_seg, _ = model(img_ss)

        # Collect images and results for batch visualization
        if lineId < top_imgs_to_be_saved:
            seg_results = pred_seg.data.max(1)[1].cpu().numpy()[:,:,:] # will be 1*H*W
            batched_img_ss.append(img_ss.cpu().data)
            batched_pred_seg.append(seg_results)
            batched_label_ss.append(label_ss.cpu().data)


        pred_seg = pred_seg.data.cpu().numpy()

        N, _, h, w = pred_seg.shape
        pred = pred_seg.transpose(0, 2, 3, 1).reshape(-1, num_class_seg).argmax(axis=1).reshape(N, h, w)

        target = label_ss.cpu().numpy().reshape(N, h, w)

        for p, t in zip(pred, target):
            total_ious.append(iou(p, t, num_class_seg))
            pixel_accs.append(pixel_acc(p, t))

    # Concatenate collected tensors for batch visualization
    if len(batched_img_ss) > 0:
        batched_img_ss = torch.cat(batched_img_ss, dim=0)
        batched_pred_seg = np.concatenate(batched_pred_seg, axis=0)
        batched_label_ss = torch.cat(batched_label_ss, dim=0)
        save_result_comparison_batched(batched_img_ss, batched_pred_seg, seg_result_test_folder, epoch, seg_class_colors, batched_label_ss)


    # Calculate average IoU
    total_ious = np.array(total_ious).T  # n_class * val_len
    ious = np.nanmean(total_ious, axis=1)
    pixel_accs = np.array(pixel_accs).mean()
    #print("epoch{}, pix_acc: {}, meanIoU: {}, IoUs: {}".format(epoch, pixel_accs, np.nanmean(ious), ious))

    return pixel_accs, ious


def test_result_lm_ss(epoch, model, test_loader, lm_seg_result_test_folder, num_class_lm_seg, lm_seg_class_colors):

    total_ious = []
    pixel_accs = []
    top_imgs_to_be_saved = 8

    # Lists to store images, predictions, and ground truth for batch visualization
    batched_img_lm_ss = []
    batched_pred_lm_seg = []
    batched_label_lm_ss = []


    for lineId, (_, _, _, _, _, img_lm_ss, label_lm_ss, one_hot_label_lm_ss) in enumerate(test_loader):

        img_lm_ss = img_lm_ss.cuda()

        _, _, _, _, pred_lm_seg = model(img_lm_ss)

        # only save the 1st image for comparison
        if lineId < top_imgs_to_be_saved:
            # generate images
            seg_results = pred_lm_seg.data.max(1)[1].cpu().numpy()[:,:,:]

            # Append single image tensors to lists
            batched_img_lm_ss.append(img_lm_ss.cpu().data)
            batched_pred_lm_seg.append(seg_results)
            batched_label_lm_ss.append(label_lm_ss.cpu().data)

        pred_seg = pred_lm_seg.data.cpu().numpy()

        N, _, h, w = pred_seg.shape
        pred = pred_seg.transpose(0, 2, 3, 1).reshape(-1, num_class_lm_seg).argmax(axis=1).reshape(N, h, w)

        target = label_lm_ss.cpu().numpy().reshape(N, h, w)

        for p, t in zip(pred, target):
            total_ious.append(iou(p, t, num_class_lm_seg))
            pixel_accs.append(pixel_acc(p, t))

    # Concatenate collected tensors for batch visualization
    if len(batched_img_lm_ss) > 0:
        batched_img_lm_ss = torch.cat(batched_img_lm_ss, dim=0)
        batched_pred_lm_seg = np.concatenate(batched_pred_lm_seg, axis=0)
        batched_label_lm_ss = torch.cat(batched_label_lm_ss, dim=0)
        save_result_comparison_batched(batched_img_lm_ss, batched_pred_lm_seg, lm_seg_result_test_folder, epoch, lm_seg_class_colors, batched_label_lm_ss)


    # Calculate average IoU
    total_ious = np.array(total_ious).T  # n_class * val_len
    ious = np.nanmean(total_ious, axis=1)
    pixel_accs = np.array(pixel_accs).mean()
    #print("epoch{}, pix_acc: {}, meanIoU: {}, IoUs: {}".format(epoch, pixel_accs, np.nanmean(ious), ious))

    return pixel_accs, ious

def evaluation_od_ss_lmss(epoch, model, prefix, multitasking_root_dir, outfile, test_list_od, test_list_ss, test_list_lm_ss, det_result_test_folder, seg_result_test_folder, lm_seg_result_test_folder, num_class_seg, num_class_lm_seg, seg_class_colors, lm_seg_class_colors, task_mode='all'):

    res_prefix = prefix + '/' + outfile # YOLOv1_BDD100K_multitasking/detection_temp_folder/VOC and the detection file will be VOCCar.txt, VOCTruck.txt etc,.

    # prepare the dataloader of the testing set.
    dataset = VOCDatasets(transform, test_list_od, test_list_ss, test_list_lm_ss, multitasking_root_dir, num_class_seg, num_class_lm_seg, seg_class_colors, lm_seg_class_colors, None, False)
    test_loader = data.DataLoader(dataset=dataset,
                                  batch_size=1,
                                  shuffle=False,
                                  num_workers=2)
    test_loader = tqdm(test_loader)


    if task_mode == 'all':
        test_result_od(epoch, model, test_loader, test_list_od, prefix, multitasking_root_dir, outfile, det_result_test_folder)
        pixel_accs, ious = test_result_ss(epoch, model, test_loader, seg_result_test_folder, num_class_seg, seg_class_colors)
        pixel_accs_lm, ious_lm = test_result_lm_ss(epoch, model, test_loader, lm_seg_result_test_folder, num_class_lm_seg, lm_seg_class_colors)

        result = _do_python_eval_quite(res_prefix, annopath, imagesetfile, output_dir = GT_temp_folder)
        return result, pixel_accs, ious, pixel_accs_lm, ious_lm
    elif task_mode == 'od':
        test_result_od(epoch, model, test_loader, test_list_od, prefix, multitasking_root_dir, outfile, det_result_test_folder)
        result = _do_python_eval_quite(res_prefix, annopath, imagesetfile, output_dir = GT_temp_folder)
        return result, np.array(0), np.array(0), np.array(0), np.array(0)
    elif task_mode == 'ss':
        pixel_accs, ious =  test_result_ss(epoch, model, test_loader, seg_result_test_folder, num_class_seg, seg_class_colors)
        return {'vehicle':0.0}, pixel_accs, ious, np.array(0), np.array(0)
    elif task_mode == 'lm_ss':
        pixel_accs_lm, ious_lm = test_result_lm_ss(epoch, model, test_loader, lm_seg_result_test_folder, num_class_lm_seg, lm_seg_class_colors)
        return {'vehicle':0.0}, np.array(0), np.array(0), pixel_accs_lm, ious_lm

In [11]:
import numpy as np
from PIL import Image

import numpy as np
from PIL import Image
import torch
import cv2
import warnings

def save_result_comparison_batched(input_np, output_np, folder_to_save_validation_result, index, seg_class_colors, GT_np=None, save_result=True):
    """
    Saves a batched comparison of input images, predicted segmentation maps, and ground truth (optional).

    Args:
        input_np (torch.Tensor or np.ndarray): Batched input images (shape: B x C x H x W).
        output_np (torch.Tensor or np.ndarray): Batched predicted segmentation maps (shape: B x H x W).
        folder_to_save_validation_result (str): Folder path to save the resulting image.
        index (int): Index to include in the filename.
        seg_class_colors (np.ndarray): Array of RGB colors for each class and it can be either free-space segmentation or lane marking segmentation.
        GT_np (torch.Tensor or np.ndarray, optional): Batched ground truth segmentation maps (shape: B x H x W). Defaults to None.
        save_result (bool, optional): Whether to save the resulting image. Defaults to True.

    Returns:
        np.ndarray: Vertically stacked comparison image for the batch.
    """
    # Mean and std for unnormalization
    means = np.array([0.485, 0.456, 0.406])
    stds = np.array([0.229, 0.224, 0.225])

    # Convert input tensor to numpy if needed
    if isinstance(input_np, torch.Tensor):
        input_np = input_np.cpu().numpy()  # Move to CPU and convert to NumPy

    # Convert output tensor to numpy if needed
    if isinstance(output_np, torch.Tensor):
        output_np = output_np.cpu().numpy()

    if GT_np is not None and isinstance(GT_np, torch.Tensor):
        GT_np = GT_np.cpu().numpy()

    # Reshape means and stds for broadcasting
    means = means[:, np.newaxis, np.newaxis] # Shape becomes (3, 1, 1)
    stds = stds[:, np.newaxis, np.newaxis]   # Shape becomes (3, 1, 1)

    batch_size, C, H, W = input_np.shape
    stacked_images = []

    for i in range(batch_size):
        # Unnormalize input image
        original_im_RGB = ((input_np[i] * stds) + means) * 255.0
        original_im_RGB = np.clip(original_im_RGB, 0, 255).astype(np.uint8)
        # Transpose to (H, W, C) for stacking
        original_im_RGB = original_im_RGB.transpose(1, 2, 0)

        # Map segmentation output to RGB colors
        im_seg_RGB = np.take(seg_class_colors, output_np[i], axis=0)

        # Check if the segmentation image is all black
        if np.all(im_seg_RGB == [0, 0, 0]):
            warnings.warn(f"Segmentation output for image {i} is all black.")

        # Check if the segmentation image is all black
        if np.all(output_np == [0]):
            warnings.warn(f"Segmentation output for predicted_segmentation {i} is all 0.")

        if GT_np is not None:
            GT_np_RGB = np.take(seg_class_colors, GT_np[i], axis=0)
            # Horizontally stack original image, predicted segmentation, and ground truth
            hstack_image = np.hstack((original_im_RGB, im_seg_RGB, GT_np_RGB))
        else:
            # Horizontally stack original image and predicted segmentation
            hstack_image = np.hstack((original_im_RGB, im_seg_RGB))

        stacked_images.append(hstack_image)

    # Vertically stack the images from the batch
    vstack_image = np.vstack(stacked_images)

    # Save image if required
    if save_result:
        file_name = f"{folder_to_save_validation_result}{index}_batched.jpg"
        Image.fromarray(vstack_image).save(file_name)

    return vstack_image

# Semantic segmentation visualization
def save_result_comparison(input_np, output_np, folder_to_save_validation_result, index, seg_class_colors, GT_np=None, save_result=True):
    # seg_class_colors (np.ndarray): Array of RGB colors for each class and it can be either free-space segmentation or lane marking segmentation.
    import torch  # For tensor type checks

    # Mean and std for unnormalization
    means = np.array([0.485, 0.456, 0.406])
    stds = np.array([0.229, 0.224, 0.225])

    # Convert input tensor to numpy if needed
    if isinstance(input_np, torch.Tensor):
        input_np = input_np.cpu().numpy()  # Move to CPU and convert to NumPy

    # Reshape means and stds for broadcasting
    means = means[:, np.newaxis, np.newaxis] # Shape becomes (3, 1, 1)
    stds = stds[:, np.newaxis, np.newaxis]   # Shape becomes (3, 1, 1)


    # Unnormalize input image
    original_im_RGB = ((input_np * stds) + means) * 255.0
    original_im_RGB = np.clip(original_im_RGB, 0, 255).astype(np.uint8)
    # Transpose to (H, W, C) for stacking
    original_im_RGB = original_im_RGB.transpose(1, 2, 0)


    # Map segmentation output to RGB colors
    im_seg_BGR = np.take(seg_class_colors, output_np, axis=0)
    im_seg_RGB = im_seg_BGR[..., ::-1]

    if GT_np is not None: # Check if GT_np is provided
        GT_np_BGR = np.take(seg_class_colors, GT_np, axis=0)
        GT_np_RGB = GT_np_BGR[..., ::-1]
        # Horizontally stack original image and segmentation results
        hstack_image = np.hstack((original_im_RGB, im_seg_RGB, GT_np_RGB))
    else:
        hstack_image = np.hstack((original_im_RGB, im_seg_RGB))



    # Save image if required
    if save_result:
        file_name = f"{folder_to_save_validation_result}{index}.jpg"
        Image.fromarray(hstack_image).save(file_name)

    return hstack_image

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Apply Softmax to the inputs to get probabilities
        inputs = F.softmax(inputs, dim=1)

        # Convert targets to one-hot encoding
        targets = F.one_hot(targets, num_classes=inputs.shape[1]).permute(0, 3, 1, 2).float()

        # Flatten label and prediction tensors
        inputs = inputs.contiguous().view(-1)
        targets = targets.contiguous().view(-1)

        intersection = (inputs * targets).sum()
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

        return 1 - dice

In [ ]:
from google.colab.patches import cv2_imshow
from collections import Counter


net = YOLO(cls_num, bbox_num, box_scale, pretrained=True, num_class_seg=num_class_seg, num_class_lm_seg=num_class_lm_seg)


lr = 0.1e-3 #0.2e-3

#else:
#    net = YOLO(cls_num, bbox_num, box_scale)
#optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
optimizer = optim.Adam(net.parameters(), lr=lr)


# # # Calculate class weights for lane marking segmentation
# # print("Calculating class weights for lane marking segmentation...")
# # pixel_counts = Counter()
# # # Iterate through the training dataset to count pixel occurrences for each class
# # for i in range(len(dataset)):
# #     # Get the label_lm_ss tensor for each image
# #     _, _, _, _, _, _, label_lm_ss, _ = dataset[i]
# #     # Flatten the tensor and update the Counter
# #     pixel_counts.update(label_lm_ss.flatten().tolist())

# # # Calculate inverse frequency weights
# # total_pixels = sum(pixel_counts.values())
# # class_weights = torch.zeros(num_class_lm_seg)
# # for class_index, count in pixel_counts.items():
# #     class_weights[class_index] = total_pixels / (count + 1e-5) # Add a small epsilon for stability

class_weights = torch.tensor([1.5232e-04, 7.2311e-02, 1.2279e-01, 7.9105e-01, 1.3699e-02])

# Normalize weights (optional, but can help with learning rate selection)
class_weights = class_weights / class_weights.sum()

print("Lane marking segmentation class weights:", class_weights)

# Changed to CrossEntropyLoss and removed BCEWithLitLoss typo
ss_criterion = nn.CrossEntropyLoss()
# Apply class weights to the lane marking segmentation loss
lm_ss_criterion_ce = nn.CrossEntropyLoss(weight=class_weights.cuda())
# Instantiate DiceLoss for lane marking segmentation
lm_ss_criterion_dice = DiceLoss()


net.cuda()
net.train()

best_score = 0
best_pixel_acc = 0
best_iou = 0
best_lm_pixel_acc = 0
best_lm_iou = 0


#eval_score = 0

loss_od_list = []
loss_ss_list = []
loss_lm_ss_list = []

for epoch in range(epochs_start, epochs_end):
    epoch_loss_od = 0
    epoch_loss_ss = 0
    epoch_loss_lm_ss = 0

    train_iterator = tqdm(train_loader, ncols=80)
    mulit_batch_ = 0
    learning_rate=lr

    for train_batch, (img_od, target, img_ss, label_ss, one_hot_label_ss, img_lm_ss, label_lm_ss, one_hot_label_lm_ss) in enumerate(train_iterator):

        if train_batch % 3 == 0: # or train_batch % 3 == 1:  # Detection task
            img_od = img_od.cuda()
            pred_cls, pred_response, pred_bboxes, _, _ = net(img_od)
            loss_xx, loss_info = loss_detect((pred_cls, pred_response, pred_bboxes), target)
            assert not np.isnan(loss_xx.data.cpu().numpy())
            epoch_loss_od += loss_xx

            status = 'epoch[{0}], iteration = {1}, lr = {2} batch_loss_od = {3:.3f} epoch_loss_od = {4:.3f} '.format(
              epoch, train_batch, learning_rate, loss_xx.data, epoch_loss_od.data / (train_batch + 1))
            loss_xx.backward()
            epoch_loss_od += loss_xx

            loss_od_list.append(loss_xx)
        elif train_batch % 3 == 1:  # Segmentation task

            img_ss = img_ss.cuda()
            # Use label_ss (class indices) as target for CrossEntropyLoss
            label_ss = label_ss.cuda()
            _, _, _, pred_seg, _ = net(img_ss)

            loss_ss = ss_criterion(pred_seg, label_ss)
            #print(f"---loss_ss={loss_ss}")
            assert not np.isnan(loss_ss.data.cpu().numpy())
            epoch_loss_ss += loss_ss
            status = 'epoch[{0}], iteration = {1}, lr = {2} batch_loss_ss = {3:.3f} epoch_loss_ss = {4:.3f} '.format(
                epoch, train_batch, learning_rate, loss_ss.data, epoch_loss_ss.data / (train_batch + 1))

            loss_ss.backward()
            loss_ss_list.append(loss_ss)

        elif train_batch % 3 == 2:  # lane marking Segmentation task

            img_lm_ss = img_lm_ss.cuda()
            # Use label_lm_ss (class indices) as target for CrossEntropyLoss
            label_lm_ss = label_lm_ss.cuda()
            _, _, _, _, pred_lm_seg = net(img_lm_ss)

            # Calculate CrossEntropyLoss and DiceLoss
            loss_lm_ss_ce = lm_ss_criterion_ce(pred_lm_seg, label_lm_ss)
            loss_lm_ss_dice = lm_ss_criterion_dice(pred_lm_seg, label_lm_ss)

            # Combine the losses (you might want to tune the weights)
            loss_lm_ss = 1.0*loss_lm_ss_ce #+ 0.8*loss_lm_ss_dice

            #print(f"---loss_ss={loss_ss}")
            assert not np.isnan(loss_lm_ss.data.cpu().numpy())
            epoch_loss_lm_ss += loss_lm_ss
            status = 'epoch[{0}], iteration = {1}, lr = {2} batch_loss_lm = {3:.3f} epoch_loss_ss_lm = {4:.3f} '.format(
                epoch, train_batch, learning_rate, loss_lm_ss.data, epoch_loss_lm_ss.data / (train_batch + 1))

            loss_lm_ss.backward()
            loss_lm_ss_list.append(loss_lm_ss)


        train_iterator.set_description(status)

        #loss_all.backward()

        optimizer.step()
        optimizer.zero_grad()



        step += 1

    if epoch % 1 == 0: # and epoch > 5:
        print("Evaluate~~~~~   ")
        net.eval()
        # Swapped num_class_lm_seg and lm_seg_class_colors in the following calls
        result, pixel_accs, ious, pixel_accs_lm, ious_lm = evaluation_od_ss_lmss(epoch, net, detection_result_txt_folder, multitasking_root_dir,'voc', eval_image_object_detection_list, eval_image_semantic_segmentation_list, eval_image_lane_marking_semantic_segmentation_list, det_result_test_folder, seg_result_test_folder, lm_seg_result_test_folder, num_class_seg, num_class_lm_seg, seg_class_colors, lm_seg_class_colors, task_mode='all')
        eval_score = np.mean(list(result.values()))

        net.train()

        if best_score < eval_score:
            best_score = eval_score
            torch.save(net.state_dict(), model_path + "best_.pkl")

        if best_iou < ious.mean():
            best_iou = ious.mean()
            torch.save(net.state_dict(), model_path + "best_iou.pkl")

        if best_pixel_acc < pixel_accs:
            best_pixel_acc = pixel_accs
            torch.save(net.state_dict(), model_path + "best_pixel_acc.pkl")

        if best_lm_iou < ious_lm.mean():
            best_lm_iou = ious_lm.mean()
            torch.save(net.state_dict(), model_path + "best_lm_iou.pkl")

        if best_lm_pixel_acc < pixel_accs_lm:
            best_lm_pixel_acc = pixel_accs_lm
            torch.save(net.state_dict(), model_path + "best_lm_pixel_acc.pkl")


        mAP_string = "ap : {:.3f} , best ap: {:.3f}, pixel_accs:{:.3f}, ious:{:.3f}, pixel_lm_accs:{:.3f}, lm_ious:{:.3f},　best iou: {:.3f}, best pixel_acc: {:.3f},　best lm iou: {:.3f}, best lm pixel_acc: {:.3f} ".format(eval_score, best_score, pixel_accs, ious.mean(), pixel_accs_lm, ious_lm.mean(), best_iou, best_pixel_acc, best_lm_iou, best_lm_pixel_acc)
        print(mAP_string)

        with open(savelog_filename, 'a') as f:
            print('%s %s' % (time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), mAP_string), file=f)

    torch.save(net.state_dict(), model_path + "model_.pkl")

print(best_score)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]


Lane marking segmentation class weights: tensor([1.5232e-04, 7.2311e-02, 1.2279e-01, 7.9105e-01, 1.3699e-02])


  0%|                                                   | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:3928: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  reduction = _Reduction.legacy_get_string(size_average, reduce)
epoch[1], iteration = 17, lr = 0.0001 batch_loss_lm = 1.618 epoch_loss_ss_lm = 0/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:3928: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  reduction = _Reduction.legacy_get_string(size_average, reduce)
epoch[1], iteration = 65, lr = 0.0001 batch_loss_lm = 1.596 epoch_loss_ss_lm = 0/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:3928: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  reduction = _Reduction.legacy_get_string(size_average, reduce)
epoch[1], iteration = 71, lr = 0.0001 batch_loss_lm = 1.585 epoc

...

100%|██████████| 99/99 [00:08<00:00, 11.94it/s]


Begin to perform mAP estimation
AP for vehicle = 0.0227
ap : 0.023 , best ap: 0.023, pixel_accs:0.921, ious:0.779, pixel_lm_accs:0.606, lm_ious:0.130,　best iou: 0.779, best pixel_acc: 0.921,　best lm iou: 0.130, best lm pixel_acc: 0.606 


epoch[5], iteration = 99, lr = 0.0001 batch_loss_od = 11.885 epoch_loss_od = 6.9


Evaluate~~~~~   


100%|██████████| 99/99 [00:50<00:00,  1.96it/s]


In [ ]:
import matplotlib.pyplot as plt

# Example list of losses
loss_od_list1 = [loss_ele.detach().cpu().numpy() for loss_ele in loss_od_list]
# Plot the losses
plt.figure(figsize=(10, 6))  # Set the figure size
plt.plot(range(len(loss_od_list1)), loss_od_list1, marker='o', linestyle='-', label='Loss')  # Plot with markers
plt.title('Detection Loss Over Iterations', fontsize=16)  # Add a title
plt.xlabel('Iteration', fontsize=14)  # Label for x-axis
plt.ylabel('Loss', fontsize=14)  # Label for y-axis
plt.grid(True)  # Add a grid for better visualization
plt.legend(fontsize=12)  # Add a legend
plt.tight_layout()  # Adjust layout for better fit
plt.show()  # Display the plot

loss_ss_list1 = [loss_ele.detach().cpu().numpy() for loss_ele in loss_ss_list]
# Plot the losses
plt.figure(figsize=(10, 6))  # Set the figure size
plt.plot(range(len(loss_ss_list1)), loss_ss_list1, marker='o', linestyle='-', label='Loss')  # Plot with markers
plt.title('Free-space Segmentation Loss Over Iterations', fontsize=16)  # Add a title
plt.xlabel('Iteration', fontsize=14)  # Label for x-axis
plt.ylabel('Loss', fontsize=14)  # Label for y-axis
plt.grid(True)  # Add a grid for better visualization
plt.legend(fontsize=12)  # Add a legend
plt.tight_layout()  # Adjust layout for better fit
plt.show()  # Display the plot

loss_lm_ss_list1 = [loss_ele.detach().cpu().numpy() for loss_ele in loss_lm_ss_list]
# Plot the losses
plt.figure(figsize=(10, 6))  # Set the figure size
plt.plot(range(len(loss_lm_ss_list1)), loss_lm_ss_list1, marker='o', linestyle='-', label='Loss')  # Plot with markers
plt.title('Lane Marking Segmentation Loss Over Iterations', fontsize=16)  # Add a title
plt.xlabel('Iteration', fontsize=14)  # Label for x-axis
plt.ylabel('Loss', fontsize=14)  # Label for y-axis
plt.grid(True)  # Add a grid for better visualization
plt.legend(fontsize=12)  # Add a legend
plt.tight_layout()  # Adjust layout for better fit
plt.show()  # Display the plot

# Detection Result Inference

In [ ]:
  #  Let's try to perform inference for single image
from google.colab.patches import cv2_imshow

def predict_gpu_od(model,image_name):

    image = cv2.imread(image_name)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h,w,_ = image.shape
    img = cv2.resize(image,(608,608))

    # Do remember that the following transform only accepts input to be numpy array instead PIL image
    # Therefore, the previously-defined transfroms() can't be used here
    transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    # image is an torch tensor
    #img = transform(img)*255 # original darknet18
    img = transform(img) # for REsnet-18
    #  None = NP.newaxis and the reason why doing this is that model inference would only accept 4-d torch tensor.
    img = Variable(img[None,:,:,:])
    img = img.cuda()

    pred_cls, pred_response, pred_bboxes, _, _= model(img)

    pred = pred_cls, pred_response, pred_bboxes
    # detection_classses contains object id and its confidence
    boxes, detection_classses = yolo_box_decoder(pred,conf_thresh=0.1,nms_thresh=0.5)

    return boxes,detection_classses



def plot_boxes_cv2(img, boxes, detection_classses, class_names=None, color=None):

    width = img.shape[1]
    height = img.shape[0]

    for j in range(len(boxes)):

        x1 = boxes[j,0]
        y1 = boxes[j,1]
        x2 = x1 + boxes[j,2]
        y2 = y1 + boxes[j,3]
        x1,x2 = int(x1*width),int(x2*width)
        y1,y2 = int(y1*height),int(y2*height)

        cls_id = int(detection_classses[j,0])
        prob = float(detection_classses[j,1])
        print(prob)

        img = cv2.putText(img, class_names[cls_id], (x1,y1),  cv2.FONT_HERSHEY_PLAIN, 1.0, (0,0,255), thickness=1)
        img = cv2.putText(img, str(round(prob,3)), (x1,y1+20),  cv2.FONT_HERSHEY_PLAIN, 1.0, (0,0,255), thickness=1)
        img = cv2.rectangle(img, (x1,y1), (x2,y2), (0, 0, 255), 2)

    return img

#model = YOLO(cls_num, bbox_num, box_scale)
model = YOLO(cls_num, bbox_num, box_scale, pretrained=True, num_class_seg=num_class_seg, num_class_lm_seg=num_class_lm_seg)
model.eval()
# load your best model trained so far.
model.load_state_dict(torch.load( model_path + 'best_.pkl'))
#model.load_state_dict(torch.load( model_path + 'model_.pkl'))


model.cuda()

# testing image
#image_name = multitasking_root_dir + "bdd100k_clear_detection_sampled_files/val/images/0f01f75e-4c950e8e.jpg"
image_name = multitasking_root_dir + "bdd100k_lane_marking_segmentation_sampled_files/val/images/003e23ee-07d32feb.jpg" # from lane seg val

image = cv2.imread(image_name)
# performing inference
boxes, detection_classses = predict_gpu_od(model,image_name)

# overlapping detection bounding box
image = plot_boxes_cv2(image, boxes, detection_classses, class_names = classes, color=None)

cv2_imshow(image)




# Free-Space Segmentation Result Inference

In [ ]:

def predict_gpu_ss(model, image_name, lm_seg_class_colors):

    image = cv2.imread(image_name)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h,w,_ = image.shape
    img = cv2.resize(image,(608,608))

    # Do remember that the following transform only accepts input to be numpy array instead PIL image
    # Therefore, the previously-defined transfroms() can't be used here
    transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    # image is an torch tensor
    #img = transform(img)*255 # original darknet18
    img = transform(img) # for REsnet-18
    #  None = NP.newaxis and the reason why doing this is that model inference would only accept 4-d torch tensor.
    img = Variable(img[None,:,:,:])
    img = img.cuda()
    #print(f"img={img.shape}")
    _, _, _, pred_seg, _= model(img)
    #pred_seg= model(img)
    print(f"pred_seg={pred_seg.shape}")

    seg_results = pred_seg.data.max(1)[1].cpu().numpy()[:,:,:]
    seg_result = seg_results[0,:,:]

    img = img.cpu().numpy()[0,:,:,:]


    segmentation_result = save_result_comparison(img, seg_result, seg_result_test_folder, 0, lm_seg_class_colors, GT_np=None, save_result = False)
    return segmentation_result

image_name = multitasking_root_dir + "bdd100k_lane_marking_segmentation_sampled_files/val/images/2e51558a-2e3724cc.jpg"

result_img = predict_gpu_ss(model, image_name, lm_seg_class_colors)
result_img_bgr = result_img[..., ::-1]
cv2_imshow(result_img_bgr)

# Lane-Marking Segmentation Result Inference

In [ ]:
def predict_gpu_lm_ss(model, image_name, lm_seg_class_colors):

    image = cv2.imread(image_name)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h,w,_ = image.shape
    img = cv2.resize(image,(608,608))

    # Do remember that the following transform only accepts input to be numpy array instead PIL image
    # Therefore, the previously-defined transfroms() can't be used here
    transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    # image is an torch tensor
    #img = transform(img)*255 # original darknet18
    img = transform(img) # for REsnet-18
    #  None = NP.newaxis and the reason why doing this is that model inference would only accept 4-d torch tensor.
    img = Variable(img[None,:,:,:])
    img = img.cuda()
    #print(f"img={img.shape}")
    _, _, _, _, pred_lm_seg= model(img)

    print(f"pred_lm_seg={pred_lm_seg.shape}")

    seg_results = pred_lm_seg.data.max(1)[1].cpu().numpy()[:,:,:]
    seg_result = seg_results[0,:,:]


    # print(sum(seg_result[seg_result==1]))
    # print(sum(seg_result[seg_result==2]))
    # print(sum(seg_result[seg_result==3]))
    # print(sum(seg_result[seg_result==4]))

    img = img.cpu().numpy()[0,:,:,:]


    segmentation_result = save_result_comparison(img, seg_result, lm_seg_result_test_folder, 0, lm_seg_class_colors, GT_np=None, save_result = False)
    return segmentation_result

model = YOLO(cls_num, bbox_num, box_scale, pretrained=True, num_class_seg=num_class_seg, num_class_lm_seg=num_class_lm_seg)
model.eval()
# load your best model trained so far.
model.load_state_dict(torch.load( model_path + 'best_lm_iou.pkl')) # best lane mearking iou model
model.cuda()



image_name = multitasking_root_dir + "bdd100k_lane_marking_segmentation_sampled_files/val/images/0464a98b-860229d5.jpg"
result_img = predict_gpu_lm_ss(model, image_name, lm_seg_class_colors)
result_img_bgr = result_img[..., ::-1]
cv2_imshow(result_img_bgr)